# LLM Efficiency — Hands-On

**LLM Engineering · Domain 2 · Roadmap Week 11**

Companion to `02 Literature Notes/LLM Engineering/LLM Efficiency`. Pure numpy, runs offline.
We make the memory math and KV-cache reuse concrete.

## 0. Setup

In [ ]:
%pip install -q numpy
import numpy as np
rng = np.random.RandomState(0)
print("ok")

## 1. KV cache can dwarf the weights at long context

In [ ]:
def weights_gb(L, d, v, b=2): return (L*12*d*d + v*d)*b/1e9
def kv_gb(L, d, seq, batch=1, b=2, kv=None, heads=None):
    frac = 1.0 if not (kv and heads) else kv/heads
    return 2*L*seq*(d*frac)*b*batch/1e9
L,d,v = 32,4096,32000
print(f"weights fp16: {weights_gb(L,d,v):.1f} GB")
for seq in (2048, 8192, 32768):
    print(f"seq={seq:>6} b=8  KV(MHA)={kv_gb(L,d,seq,8):6.1f} GB  KV(GQA 8/32)={kv_gb(L,d,seq,8,kv=8,heads=32):6.1f} GB")

> At 32k context / batch 8, MHA KV cache exceeds the model weights. GQA cuts it 4x.

## 2. KV cache reuse: O(seq) vs O(seq²)

In [ ]:
def work_no_cache(n):  return sum(range(1, n+1))   # rebuild 0..t each step
def work_cache(n):     return n                    # append one per step
for n in (64, 256, 1024):
    print(f"seq={n:>5}  no-cache work={work_no_cache(n):>8}  cache work={work_cache(n):>5}  speedup~{work_no_cache(n)//work_cache(n)}x")

## 3. Quantization: memory vs precision

In [ ]:
w = rng.randn(1000).astype(np.float32)
def quantize(x, bits):
    lo, hi = x.min(), x.max(); levels = 2**bits - 1
    q = np.round((x - lo) / (hi - lo) * levels)
    deq = q / levels * (hi - lo) + lo
    return deq, x.nbytes, x.nbytes * bits // 32
for bits in (8, 4):
    deq, orig, approx = quantize(w, bits)
    err = np.abs(deq - w).mean()
    print(f"{bits}-bit: bytes {orig}->{approx} ({orig//approx}x smaller)  mean abs err={err:.4f}")

## 4. MoE: params up, per-token compute flat

In [ ]:
def moe(n_experts, top_k, d, ff):
    total_params = n_experts * 2*d*ff
    active_params = top_k * 2*d*ff
    return total_params, active_params
for ne in (8, 64):
    tot, act = moe(ne, 2, 4096, 16384)
    print(f"{ne} experts, top-2:  total={tot/1e9:.1f}B params, active/token={act/1e9:.1f}B  ({tot//act}x capacity at same compute)")

## 5. Exercises
1. Recompute KV cache for GQA with kv_heads=1 (pure MQA). How much smaller?
2. Plot mean quant error vs bits (2,3,4,8). Where does it get painful?
3. Estimate total VRAM = weights + KV + activations for a 7B model at seq=8k, batch=4.
4. Vary MoE top_k and see the capacity/compute tradeoff shift.

## Links
- Literature note: `02 Literature Notes/LLM Engineering/LLM Efficiency`
- Snippets: `04 Code Snippets/LLM/KV Cache Memory Estimator`, `.../KV Cache Reuse Demo`
- MOC: `06 Maps of Content/LLM Engineering Concepts`